In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q flask pyngrok faiss-cpu sentence-transformers transformers accelerate pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 74.7 MB/s eta 0:00:00:00:0100:01


In [3]:
import io, re, json, uuid, math, numpy as np, faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
DOCS = {}  # doc_id -> {"chunks": [...], "index": faiss index, "cards": [...]}

def pdf_to_text(file_bytes):
    reader = PdfReader(io.BytesIO(file_bytes))
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def chunk_text(text, size=1500, overlap=200):
    text = re.sub(r"\s+", " ", text).strip()
    return [text[i:i+size] for i in range(0, len(text), size - overlap) if text[i:i+size].strip()]

def ingest(file_bytes):
    chunks = chunk_text(pdf_to_text(file_bytes))
    if not chunks:
        raise ValueError("No text found in this PDF. It may be a scanned image (needs OCR).")
    emb = embedder.encode(chunks, normalize_embeddings=True).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    doc_id = uuid.uuid4().hex[:8]
    DOCS[doc_id] = {"chunks": chunks, "index": index, "cards": []}
    return doc_id, len(chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
import torch
from transformers import pipeline

llm = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct",
               torch_dtype=torch.float16, device_map="auto")

CARD_PROMPT = """You are a study assistant. From the text below, write {n} flashcards.
Each card tests one important fact or concept. Questions must be self-contained.
Answers must be short (1-2 sentences) and based only on the text.
Return ONLY a JSON list like: [{{"q": "...", "a": "..."}}]

Text:
{text}"""

def parse_cards(raw):
    m = re.search(r"\[.*\]", raw, re.DOTALL)
    if not m: return []
    try:
        return [c for c in json.loads(m.group(0)) if "q" in c and "a" in c]
    except json.JSONDecodeError:
        return []

def evenly(items, k):
    """Pick k items spread evenly across the list so the whole PDF stays covered."""
    if k >= len(items): return list(items)
    idx = sorted(set(np.linspace(0, len(items) - 1, k).round().astype(int)))
    return [items[i] for i in idx]

def generate_cards(doc_id, target=30):
    doc = DOCS[doc_id]
    use = evenly(doc["chunks"], target)            # if target < #chunks, use spaced-out chunks
    per_chunk = max(1, math.ceil(target / len(use)))
    cards, seen = [], set()
    for ch in use:
        msgs = [{"role": "user", "content": CARD_PROMPT.format(n=per_chunk, text=ch)}]
        raw = llm(msgs, max_new_tokens=120 * per_chunk + 100,
                  do_sample=False)[0]["generated_text"][-1]["content"]
        for c in parse_cards(raw):
            if c["q"].lower() not in seen:
                seen.add(c["q"].lower()); cards.append(c)
    cards = evenly(cards, target)                  # trim to target without favouring early chunks
    doc["cards"] = cards
    return cards

def grade(student_answer, reference, question=""):
    """Returns (similarity_score, is_correct)."""
    if not student_answer.strip():
        return 0.0, False
    v = embedder.encode([student_answer, reference], normalize_embeddings=True)
    score = float(v[0] @ v[1])
    if score >= 0.75: return score, True       # clearly close
    if score < 0.35:  return score, False      # clearly unrelated
    # Borderline: ask the LLM to judge meaning
    prompt = f"""Question: {question}
Reference answer: {reference}
Student answer: {student_answer}

Does the student's answer convey the same key idea as the reference answer? Be lenient about wording, but strict about facts. Reply with only CORRECT or INCORRECT."""
    out = llm([{"role": "user", "content": prompt}], max_new_tokens=5,
              do_sample=False)[0]["generated_text"][-1]["content"]
    return score, out.strip().upper().startswith("CORRECT")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [7]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
from werkzeug.serving import make_server
import threading

app = Flask(__name__)

PAGE = r"""<!DOCTYPE html>
<html lang="en"><head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Study Buddy</title>
<link href="https://fonts.googleapis.com/css2?family=Nunito:wght@600;800;900&display=swap" rel="stylesheet">
<style>
:root{--purple:#6c5ce7;--pink:#fd79a8;--green:#00b894;--red:#ff7675;--yellow:#fdcb6e;--blue:#0984e3;--ink:#2d3436}
*{box-sizing:border-box}
body{margin:0;min-height:100vh;font-family:'Nunito','Trebuchet MS',system-ui,sans-serif;color:var(--ink);
  background:linear-gradient(135deg,#a29bfe 0%,#fd79a8 50%,#fdcb6e 100%);background-attachment:fixed;padding:24px 14px}
.wrap{max-width:680px;margin:0 auto}
h1{color:#fff;text-align:center;font-weight:900;font-size:2.3rem;margin:0 0 4px;text-shadow:0 3px 0 rgba(0,0,0,.15)}
.sub{color:#fff;text-align:center;font-weight:700;margin:0 0 20px}
h2,h3{margin:8px 0}
.panel{background:#fff;border-radius:24px;padding:22px;box-shadow:0 10px 0 rgba(0,0,0,.12);margin-bottom:16px}
.hidden{display:none!important}
.row{display:flex;gap:10px;align-items:center;flex-wrap:wrap}
.between{justify-content:space-between}
.center{justify-content:center;text-align:center}
.small{font-weight:600;opacity:.75;font-size:.9rem}
.hint{text-align:center;font-weight:700;opacity:.65;margin:6px 0 14px;font-size:.9rem}
.hint.white{color:#fff;opacity:.95}

button{font-family:inherit;font-weight:800;font-size:1rem;border:0;border-radius:999px;padding:11px 20px;cursor:pointer;
  color:#fff;background:var(--purple);box-shadow:0 4px 0 rgba(0,0,0,.18);transition:transform .1s,box-shadow .1s}
button:hover{transform:translateY(-2px)}
button:active{transform:translateY(3px);box-shadow:0 1px 0 rgba(0,0,0,.18)}
button:disabled{opacity:.5;cursor:not-allowed;transform:none}
button.green{background:var(--green)} button.red{background:var(--red)} button.blue{background:var(--blue)}
button.pink{background:var(--pink)} button.yellow{background:var(--yellow);color:var(--ink)}
input[type=text],input[type=number]{font-family:inherit;font-weight:700;font-size:1rem;border:3px solid #dfe6e9;
  border-radius:14px;padding:10px 14px;outline:none}
input:focus{border-color:var(--purple)}

label.drop{display:block;border:3px dashed var(--purple);border-radius:18px;padding:26px;text-align:center;
  font-weight:800;cursor:pointer;background:#f7f5ff;margin-bottom:14px}
label.drop:hover{background:#efeaff}
#status{margin-top:14px;font-weight:800;text-align:center}
.bounce{display:inline-block;animation:bounce .8s infinite}
@keyframes bounce{0%,100%{transform:translateY(0)}50%{transform:translateY(-8px)}}
@keyframes pop{0%{transform:scale(.92);opacity:0}100%{transform:scale(1);opacity:1}}

/* flip cards */
.scene{perspective:1000px;cursor:pointer;margin:14px 0}
.flipper{position:relative;height:240px;transition:transform .6s;transform-style:preserve-3d}
.flipper.flipped{transform:rotateY(180deg)}
.face{position:absolute;inset:0;border-radius:20px;backface-visibility:hidden;-webkit-backface-visibility:hidden;
  display:flex;flex-direction:column;align-items:center;justify-content:center;padding:22px;text-align:center;color:#fff;overflow:auto}
.face p{font-size:1.2rem;font-weight:800;margin:8px 0 0}
.face small{letter-spacing:2px;opacity:.85;font-weight:800}
.front{background:linear-gradient(135deg,#6c5ce7,#a29bfe)}
.back{background:linear-gradient(135deg,#00b894,#55efc4);transform:rotateY(180deg)}

/* quiz */
.pill{background:#f1f2f6;border-radius:999px;padding:6px 14px;font-weight:800;font-size:.9rem}
.pill.g{background:#d5f5ec;color:#00785f}.pill.r{background:#ffe3e3;color:#c0392b}.pill.y{background:#fff3d1;color:#9a6700}
.track{height:14px;background:#eee;border-radius:99px;overflow:hidden;margin:12px 0}
.fill{height:100%;width:0;background:linear-gradient(90deg,#55efc4,#00b894);border-radius:99px;transition:width .4s}
.fill.timer{background:linear-gradient(90deg,#74b9ff,#0984e3);transition:width .1s linear}
.fill.timer.low{background:linear-gradient(90deg,#ff7675,#d63031)}
.timerRow{display:flex;align-items:center;gap:12px}
.timerRow .track{flex:1;margin:0}
#timerText{font-weight:900;min-width:64px}
#qText{font-size:1.35rem;margin:18px 0 12px}
.feedback{margin-top:14px;padding:14px;border-radius:16px;font-weight:700;animation:pop .3s}
.feedback.good{background:#d5f5ec}.feedback.bad{background:#ffe3e3}
.feedback button{margin-top:10px}

/* summary */
.stats{display:grid;grid-template-columns:repeat(auto-fit,minmax(120px,1fr));gap:10px;margin:16px 0}
.stat{background:#f7f5ff;border-radius:16px;padding:12px}
.stat b{display:block;font-size:1.4rem}.stat small{font-weight:700;opacity:.65}
ul.missed{text-align:left;padding-left:20px}
ul.missed li{margin-bottom:10px}
ul.missed span{opacity:.75;font-weight:600}
</style></head>
<body><div class="wrap">
<h1>🧠 Study Buddy</h1>
<p class="sub">Upload your notes, flip through the cards, then beat the clock!</p>

<!-- UPLOAD -->
<section id="scrUpload"><div class="panel">
  <label class="drop"><input type="file" id="pdf" accept=".pdf" hidden><span id="fname">📄 Choose a PDF</span></label>
  <div class="row between">
    <label>🃏 Cards <input type="number" id="target" value="30" min="5" max="100" style="width:80px"></label>
    <button id="makeBtn" class="green" onclick="makeCards()">Make my cards ✨</button>
  </div>
  <div id="status"></div>
</div></section>

<!-- STUDY -->
<section id="scrStudy" class="hidden">
  <div class="panel">
    <div class="row between"><b>📚 Study mode</b><span id="sPos" class="pill"></span></div>
    <div class="scene" onclick="flipCard()"><div class="flipper" id="flipper">
      <div class="face front"><small>QUESTION</small><p id="sQ"></p></div>
      <div class="face back"><small>ANSWER</small><p id="sA"></p></div>
    </div></div>
    <p class="hint">Click the card or press Space to flip · ← → to move</p>
    <div class="row center">
      <button class="blue" onclick="studyMove(-1)">◀ Prev</button>
      <button class="yellow" onclick="flipCard()">🔄 Flip</button>
      <button class="blue" onclick="studyMove(1)">Next ▶</button>
    </div>
    <div class="row center" style="margin-top:14px">
      <button class="pink" onclick="shuffleStudy()">🔀 Shuffle</button>
      <button class="pink" onclick="showScreen('Upload')">📄 New PDF</button>
    </div>
  </div>
  <div class="panel row between">
    <label>⏱ Seconds per card <input type="number" id="limit" value="30" min="0" max="300" style="width:80px"></label>
    <button class="green" onclick="startQuiz()">Start quiz 🚀</button>
  </div>
  <p class="hint white">Set seconds to 0 for no time limit</p>
</section>

<!-- QUIZ -->
<section id="scrQuiz" class="hidden"><div class="panel">
  <div class="row between">
    <span class="pill" id="qPos"></span>
    <span class="pill g" id="scorePill">✅ 0</span>
    <span class="pill r" id="missPill">❌ 0</span>
    <span class="pill y" id="streakPill">🔥 0</span>
  </div>
  <div class="track"><div class="fill" id="progress"></div></div>
  <div class="timerRow"><span id="timerText">⏱ 30s</span>
    <div class="track" id="timerTrack"><div class="fill timer" id="timerBar"></div></div></div>
  <h2 id="qText"></h2>
  <input type="text" id="answer" autocomplete="off" placeholder="Type your answer..." style="width:100%">
  <div class="row" style="margin-top:12px">
    <button class="green" id="submitBtn" onclick="submitAnswer(false)">Submit ✔</button>
    <button class="red" onclick="exitTest()">Exit test</button>
  </div>
  <div id="feedback" class="feedback hidden"></div>
</div></section>

<!-- SUMMARY -->
<section id="scrSummary" class="hidden"><div class="panel center" style="display:block">
  <div id="bigEmoji" style="font-size:4rem;text-align:center">🎉</div>
  <h2 id="sumTitle" style="text-align:center"></h2>
  <p id="sumNote" class="small" style="text-align:center"></p>
  <div class="stats">
    <div class="stat"><b id="stScore"></b><small>Score</small></div>
    <div class="stat"><b id="stAvg"></b><small>Avg per card</small></div>
    <div class="stat"><b id="stTotal"></b><small>Total time</small></div>
    <div class="stat"><b id="stStreak"></b><small>Best streak</small></div>
  </div>
  <div id="missedBox"></div>
  <div class="row center" id="sumButtons" style="margin-top:16px"></div>
</div></section>
</div>

<script>
let allCards=[], studyCards=[], cards=[], missedCards=[];
let s=0, i=0, results=[], limit=30, t0=0, tick=null, nextTimer=null;
let busy=false, awaitingNext=false, streak=0, bestStreak=0, correctCount=0, sessionStart=0, runId=0;
const $=id=>document.getElementById(id);
const esc=x=>String(x).replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
function showScreen(n){ ['Upload','Study','Quiz','Summary'].forEach(k=>$('scr'+k).classList.toggle('hidden',k!==n)); }
function shuffle(a){ a=a.slice(); for(let k=a.length-1;k>0;k--){const j=Math.floor(Math.random()*(k+1)); [a[k],a[j]]=[a[j],a[k]];} return a; }

$('pdf').addEventListener('change',()=>{ const f=$('pdf').files[0]; $('fname').textContent=f?('📄 '+f.name):'📄 Choose a PDF'; });

/* ---------- upload ---------- */
async function makeCards(){
  const f=$('pdf').files[0];
  if(!f){ $('status').textContent='Pick a PDF first 👆'; return; }
  $('makeBtn').disabled=true;
  $('status').innerHTML='<span class="bounce">🧠</span> Reading your PDF and cooking up cards... this can take a few minutes';
  const fd=new FormData(); fd.append('pdf',f); fd.append('target',$('target').value);
  try{
    const d=await (await fetch('/upload',{method:'POST',body:fd})).json();
    if(d.error){ $('status').textContent='😕 '+d.error; return; }
    if(!d.cards.length){ $('status').textContent='😕 No cards came out. Try another PDF or check the notebook output.'; return; }
    allCards=d.cards; studyCards=allCards.slice(); s=0; renderStudy();
    $('status').textContent=''; showScreen('Study');
  }catch(e){ $('status').textContent='😕 Request failed. Check the notebook output for errors.'; }
  finally{ $('makeBtn').disabled=false; }
}

/* ---------- study mode ---------- */
function renderStudy(){
  const wasFlipped=$('flipper').classList.contains('flipped');
  $('flipper').classList.remove('flipped');
  $('sPos').textContent='Card '+(s+1)+' / '+studyCards.length;
  $('sQ').textContent=studyCards[s].q;
  setTimeout(()=>{ $('sA').textContent=studyCards[s].a; }, wasFlipped?300:0);
}
function flipCard(){ $('flipper').classList.toggle('flipped'); }
function studyMove(d){ s=(s+d+studyCards.length)%studyCards.length; renderStudy(); }
function shuffleStudy(){ studyCards=shuffle(studyCards); s=0; renderStudy(); }

/* ---------- quiz ---------- */
function startQuiz(){
  limit=Math.max(0,parseInt($('limit').value)||0);
  runTest(allCards);
}
function runTest(pool){
  runId++; clearInterval(tick); clearTimeout(nextTimer);
  cards=shuffle(pool); i=0; results=[]; streak=0; bestStreak=0; correctCount=0;
  busy=false; awaitingNext=false; sessionStart=Date.now();
  $('timerTrack').style.visibility = limit>0 ? 'visible' : 'hidden';
  showScreen('Quiz'); showQuestion();
}
function updatePills(){
  $('scorePill').textContent='✅ '+correctCount;
  $('missPill').textContent='❌ '+(results.length-correctCount);
  $('streakPill').textContent='🔥 '+streak;
}
function showQuestion(){
  busy=false; awaitingNext=false;
  $('qPos').textContent='Card '+(i+1)+' / '+cards.length;
  $('qText').textContent=cards[i].q;
  $('answer').value=''; $('answer').disabled=false; $('submitBtn').disabled=false;
  $('feedback').classList.add('hidden');
  updatePills();
  $('progress').style.width=(i/cards.length*100)+'%';
  $('answer').focus();
  t0=Date.now(); clearInterval(tick);
  tick=setInterval(onTick,100); onTick();
}
function onTick(){
  const el=(Date.now()-t0)/1000;
  if(limit>0){
    const rem=Math.max(0,limit-el);
    $('timerText').textContent='⏱ '+Math.ceil(rem)+'s';
    $('timerBar').style.width=(rem/limit*100)+'%';
    $('timerBar').classList.toggle('low', rem<=Math.min(5,limit*0.25));
    if(rem<=0) submitAnswer(true);
  } else {
    $('timerText').textContent='⏱ '+Math.floor(el)+'s';
  }
}
async function submitAnswer(timedOut){
  if(busy) return; busy=true;
  const myRun=runId;
  clearInterval(tick);
  let secs=(Date.now()-t0)/1000; if(limit>0) secs=Math.min(secs,limit);
  const ans=$('answer').value; $('answer').disabled=true; $('submitBtn').disabled=true;
  const card=cards[i];
  let ok=false, score=0;
  try{
    const g=await (await fetch('/grade',{method:'POST',headers:{'Content-Type':'application/json'},
      body:JSON.stringify({answer:ans,reference:card.a,question:card.q})})).json();
    ok=!!g.ok; score=g.score||0;
  }catch(e){ ok=false; }
  if(myRun!==runId) return;   // test was exited or restarted while grading
  results.push({q:card.q,a:card.a,secs,ok});
  if(ok){ correctCount++; streak++; bestStreak=Math.max(bestStreak,streak); } else { streak=0; }
  updatePills();
  $('progress').style.width=((i+1)/cards.length*100)+'%';
  const head = ok ? '🎉 Nice one!' : (timedOut ? "⏰ Time's up!" : '❌ Not quite');
  const fb=$('feedback'); fb.className='feedback '+(ok?'good':'bad');
  fb.innerHTML='<b>'+head+'</b> <span class="small">('+secs.toFixed(1)+'s, similarity '+score.toFixed(2)+')</span><br>Answer: '+
    esc(card.a)+'<br><button class="blue" onclick="advance()">Next ➜</button>';
  awaitingNext=true;
  nextTimer=setTimeout(advance,3000);
}
function advance(){
  if(!awaitingNext) return;
  awaitingNext=false; clearTimeout(nextTimer);
  i++; if(i<cards.length) showQuestion(); else finish(false);
}
function exitTest(){
  runId++; clearInterval(tick); clearTimeout(nextTimer); busy=false; awaitingNext=false;
  if(results.length>0) finish(true); else showScreen('Study');
}

/* ---------- summary ---------- */
function finish(early){
  clearInterval(tick); clearTimeout(nextTimer);
  const n=results.length, correct=results.filter(r=>r.ok).length;
  const avg=results.reduce((a,r)=>a+r.secs,0)/n, total=(Date.now()-sessionStart)/1000, pct=correct/n;
  missedCards=results.filter(r=>!r.ok).map(r=>({q:r.q,a:r.a}));
  $('bigEmoji').textContent = pct===1?'🏆':pct>=0.8?'🎉':pct>=0.5?'💪':'📚';
  $('sumTitle').textContent = pct===1?'Perfect run!':pct>=0.8?'Great job!':pct>=0.5?'Good effort!':'Keep practising!';
  $('sumNote').textContent = early ? ('Exited early: '+n+' of '+cards.length+' answered') : '';
  $('stScore').textContent=correct+'/'+n;
  $('stAvg').textContent=avg.toFixed(1)+'s';
  $('stTotal').textContent=Math.round(total)+'s';
  $('stStreak').textContent='🔥 '+bestStreak;
  $('missedBox').innerHTML = missedCards.length
    ? '<h3>Revise these again</h3><ul class="missed">'+missedCards.map(m=>'<li><b>'+esc(m.q)+'</b><br><span>'+esc(m.a)+'</span></li>').join('')+'</ul>'
    : '<h3>No misses 🎉</h3>';
  $('sumButtons').innerHTML =
    (missedCards.length ? '<button class="yellow" onclick="retakeMissed()">🎯 Retake missed only ('+missedCards.length+')</button>' : '')+
    '<button class="green" onclick="retakeFull()">🔁 Retake test</button>'+
    '<button class="blue" onclick="backToStudy()">📚 Study cards</button>';
  showScreen('Summary');
}
function retakeFull(){ runTest(allCards); }
function retakeMissed(){ runTest(missedCards); }
function backToStudy(){ studyCards=allCards.slice(); s=0; renderStudy(); showScreen('Study'); }

/* ---------- keyboard ---------- */
document.addEventListener('keydown',e=>{
  const tag=document.activeElement.tagName;
  if(!$('scrStudy').classList.contains('hidden') && tag!=='INPUT'){
    if(e.key==='ArrowRight') studyMove(1);
    else if(e.key==='ArrowLeft') studyMove(-1);
    else if(e.key===' '){ e.preventDefault(); flipCard(); }
  }
  if(!$('scrQuiz').classList.contains('hidden') && e.key==='Enter' && tag!=='BUTTON'){
    e.preventDefault();
    awaitingNext ? advance() : submitAnswer(false);
  }
});
</script></body></html>"""

@app.route("/")
def home():
    return PAGE   # returned as-is (no Jinja), so curly braces in the JS/CSS are safe

@app.route("/upload", methods=["POST"])
def upload():
    try:
        target = max(1, min(int(request.form.get("target", 30)), 100))
        doc_id, n = ingest(request.files["pdf"].read())
        cards = generate_cards(doc_id, target=target)
        return jsonify(doc_id=doc_id, chunks=n, cards=cards)
    except Exception as e:
        return jsonify(error=str(e)), 400

@app.route("/grade", methods=["POST"])
def grade_route():
    d = request.json
    score, ok = grade(d["answer"], d["reference"], d.get("question", ""))
    return jsonify(score=score, ok=ok)

# Stop the previous server if this cell was run before
try:
    server.shutdown()
except NameError:
    pass

ngrok.kill()
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))
print("Open:", ngrok.connect(5000).public_url)

server = make_server("0.0.0.0", 5000, app)
threading.Thread(target=server.serve_forever, daemon=True).start()

Open: https://little-railroad-commotion.ngrok-free.dev


127.0.0.1 - - [08/Oct/2026 22:19:57] "GET / HTTP/1.1" 200 -
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the 

In [6]:
print(grade("It can solve problems that are too complex to write rules for by hand.",
            "Machine learning can handle problems where writing rules by hand is impractical."))

0.6659468412399292
